In [1]:
import pandas as pd

In [2]:
dataset = pd.read_csv("final_cleaned_data.csv")

what is RFM_analysis?                                                      
R stand for "Recency" - it shows last purchase ofcustomer.                              
F stand for "Frequency" - it shows how often customer purchase.                                   
M stand for "Monetary" - it shows how much customer spend.                                 

First we have to find snapshotdate(reference date) for recency.

In [3]:
dataset["InvoiceDate"] = pd.to_datetime(dataset["InvoiceDate"])

In [4]:
snapshot_date = dataset["InvoiceDate"].max() + pd.Timedelta(days=1)

In [5]:
rfm = dataset.groupby("Customer ID").agg({
    "InvoiceDate" : lambda x :(snapshot_date - x.max()).days,
    "Invoice" : "nunique",
    "Total_amount" : "sum"
})

In [6]:
rfm.columns = ["Recency","Frequency","Monetary"]

In [7]:
rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,165,11,372.86
12347.0,3,2,1323.32
12348.0,74,1,222.16
12349.0,43,3,2671.14
12351.0,11,1,300.93


# RFM scaling

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm)


In [9]:
rfm_scaled_df = pd.DataFrame(rfm_scaled,index=rfm.index,columns=rfm.columns)
rfm_scaled_df.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,0.762299,0.801087,-0.187961
12347.0,-0.910402,-0.300603,-0.081329
12348.0,-0.177305,-0.423013,-0.204868
12349.0,-0.497389,-0.178193,0.069883
12351.0,-0.827799,-0.423013,-0.196031


In [10]:
rfm_scaled_df.to_csv("final_model_data.csv")

In [11]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [12]:
for k in range(2,11):
    knn = KMeans(n_clusters=k,random_state=42)
    knn.fit(rfm_scaled_df)
    score = silhouette_score(rfm_scaled_df,labels=knn.labels_)
    print(score,k)

0.5641225309439145 2
0.5890343852428707 3
0.6099377255584927 4
0.44654779608418466 5
0.5050293216690563 6
0.499286021748111 7
0.4945955716186266 8
0.46167706978214573 9
0.46168322787532245 10


In [13]:
model = KMeans(n_clusters=4,random_state=42)
rfm_scaled_df["cluster"] = model.fit_predict(rfm_scaled_df)

In [14]:
rfm_scaled_df.head(10)

,Recency,Frequency,Monetary,cluster
Customer ID,,,,
12346.0,0.762299,0.801087,-0.187961,1
12347.0,-0.910402,-0.300603,-0.081329,0
12348.0,-0.177305,-0.423013,-0.204868,0
12349.0,-0.497389,-0.178193,0.069883,0
12351.0,-0.827799,-0.423013,-0.196031,0
12352.0,-0.827799,-0.300603,-0.191221,0
12353.0,-0.487064,-0.423013,-0.194142,0
12355.0,1.154660,-0.423013,-0.175020,1
12356.0,-0.776173,-0.178193,0.169857,0


In [15]:
rfm_scaled_df["cluster"].value_counts()

cluster
0    3207
1    1047
3      53
2       5
Name: count, dtype: int64

In [16]:
rfm_scaled_df.groupby('cluster')[["Recency","Frequency","Monetary"]].mean()


,Recency,Frequency,Monetary
cluster,,,
0,-0.497421,0.001701,-0.034230
1,1.567426,-0.342225,-0.162827
2,-0.883556,13.360355,23.952060
3,-0.782017,5.397237,3.028227


In [17]:
cluster_names = {
    0 : "regular Customer",
    1 : "Lost Customer",
    2 : "VIP Customer",
    3 : "Loyal customer"
}

In [18]:
rfm_scaled_df["cluster_name"] = rfm_scaled_df["cluster"].map(cluster_names)

In [26]:
rfm_scaled_df["cluster_name"].value_counts()

cluster_name
regular Customer    3207
Lost Customer       1047
Loyal customer        53
VIP Customer           5
Name: count, dtype: int64

In [20]:
rfm_scaled_df.to_csv("RFM.csv")

In [21]:
dataset1 = pd.read_csv("final_cleaned_data.csv")

In [22]:
dataset1.head(5)

,Unnamed: 0,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total_amount
0,0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [23]:
dataset.head(5)

,Unnamed: 0,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total_amount
0,0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [24]:
new_dataset = pd.merge(dataset1,rfm_scaled_df,on="Customer ID",how="outer")

In [25]:
new_dataset.head()

,Unnamed: 0,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total_amount,Recency,Frequency,Monetary,cluster,cluster_name
0,27994,491725,TEST001,This is a test product.,10,2009-12-14 08:34:00,4.5,12346.0,United Kingdom,45.0,0.762299,0.801087,-0.187961,1,Lost Customer
1,28251,491742,TEST001,This is a test product.,5,2009-12-14 11:00:00,4.5,12346.0,United Kingdom,22.5,0.762299,0.801087,-0.187961,1,Lost Customer
2,28254,491744,TEST001,This is a test product.,5,2009-12-14 11:02:00,4.5,12346.0,United Kingdom,22.5,0.762299,0.801087,-0.187961,1,Lost Customer
3,39398,492718,TEST001,This is a test product.,5,2009-12-18 10:47:00,4.5,12346.0,United Kingdom,22.5,0.762299,0.801087,-0.187961,1,Lost Customer
4,39411,492722,TEST002,This is a test product.,1,2009-12-18 10:55:00,1.0,12346.0,United Kingdom,1.0,0.762299,0.801087,-0.187961,1,Lost Customer
